In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession
        .builder
        .appName("spark-local-app")
        .master("local[*]")
        .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/18 19:09:02 WARN Utils: Your hostname, DESKTOP-440A2SS, resolves to a loopback address: 127.0.1.1; using 172.20.131.26 instead (on interface eth0)
26/10/18 19:09:02 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/tem/de_lrn_prcts/.venv/lib/python3.14/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/10/18 19:09:14 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [5]:
users = [
    {"user_id": 101, "name": "Алексей", "city": "Москва"},
    {"user_id": 102, "name": "Мария", "city": "Санкт-Петербург"},
    {"user_id": 103, "name": "Иван", "city": "Новосибирск"},
    {"user_id": 104, "name": "Елена", "city": "Екатеринбург"},
    {"user_id": 105, "name": "Дмитрий", "city": "Казань"}
]

courses = [
    {"course_id": 1, "title": "Python для начинающих", "price": 4500},
    {"course_id": 2, "title": "Data Science и ИИ", "price": 12000},
    {"course_id": 3, "title": "Веб-разработка на Django", "price": 7500}
]

enrollments = [
    {"user_id": 101, "course_id": 1, "date": "2026-01-10"},
    {"user_id": 101, "course_id": 2, "date": "2026-02-15"},
    {"user_id": 102, "course_id": 2, "date": "2026-01-20"},
    {"user_id": 103, "course_id": 1, "date": "2026-03-01"},
    {"user_id": 104, "course_id": 3, "date": "2026-02-10"},
    {"user_id": 104, "course_id": 1, "date": "2026-03-05"},
    {"user_id": 105, "course_id": 99, "date": "2026-01-05"}
]

In [6]:
df_users = spark.createDataFrame(users)
df_courses = spark.createDataFrame(courses)
df_enrollments = spark.createDataFrame(enrollments)

In [7]:
(
    df_users
        .join(df_enrollments, df_users.user_id == df_enrollments.user_id)
        .show()
)

+---------------+-------+-------+---------+----------+-------+
|           city|   name|user_id|course_id|      date|user_id|
+---------------+-------+-------+---------+----------+-------+
|         Москва|Алексей|    101|        1|2026-01-10|    101|
|         Москва|Алексей|    101|        2|2026-02-15|    101|
|Санкт-Петербург|  Мария|    102|        2|2026-01-20|    102|
|    Новосибирск|   Иван|    103|        1|2026-03-01|    103|
|   Екатеринбург|  Елена|    104|        3|2026-02-10|    104|
|   Екатеринбург|  Елена|    104|        1|2026-03-05|    104|
|         Казань|Дмитрий|    105|       99|2026-01-05|    105|
+---------------+-------+-------+---------+----------+-------+



In [8]:
(
    df_users
        .join(df_enrollments, "user_id")
        .show()
)

+-------+---------------+-------+---------+----------+
|user_id|           city|   name|course_id|      date|
+-------+---------------+-------+---------+----------+
|    101|         Москва|Алексей|        1|2026-01-10|
|    101|         Москва|Алексей|        2|2026-02-15|
|    102|Санкт-Петербург|  Мария|        2|2026-01-20|
|    103|    Новосибирск|   Иван|        1|2026-03-01|
|    104|   Екатеринбург|  Елена|        3|2026-02-10|
|    104|   Екатеринбург|  Елена|        1|2026-03-05|
|    105|         Казань|Дмитрий|       99|2026-01-05|
+-------+---------------+-------+---------+----------+



In [15]:
(
    df_users
        .join(df_enrollments, "user_id")
        .select(df_users["*"], df_enrollments.course_id.alias("course"))
        .withColumnRenamed("name", "first_name")
        .show()
)

+---------------+----------+-------+------+
|           city|first_name|user_id|course|
+---------------+----------+-------+------+
|         Москва|   Алексей|    101|     1|
|         Москва|   Алексей|    101|     2|
|Санкт-Петербург|     Мария|    102|     2|
|    Новосибирск|      Иван|    103|     1|
|   Екатеринбург|     Елена|    104|     3|
|   Екатеринбург|     Елена|    104|     1|
|         Казань|   Дмитрий|    105|    99|
+---------------+----------+-------+------+



In [16]:
(
    df_users
        .join(df_enrollments, "user_id")
        .groupBy("user_id")
        .count()
        .show()
)

+-------+-----+
|user_id|count|
+-------+-----+
|    101|    2|
|    102|    1|
|    103|    1|
|    104|    2|
|    105|    1|
+-------+-----+



In [18]:
(
    df_users.alias("u")
        .join(df_enrollments.alias("e"), "user_id")
        .select("u.name", "u.user_id", "e.course_id")
        .show()
)

+-------+-------+---------+
|   name|user_id|course_id|
+-------+-------+---------+
|Алексей|    101|        1|
|Алексей|    101|        2|
|  Мария|    102|        2|
|   Иван|    103|        1|
|  Елена|    104|        3|
|  Елена|    104|        1|
|Дмитрий|    105|       99|
+-------+-------+---------+



In [19]:
(
    df_users.alias("u")
        .join(df_enrollments.alias("e"), "user_id", "left")
        .select("u.name", "u.user_id", "e.course_id")
        .show()
)

+-------+-------+---------+
|   name|user_id|course_id|
+-------+-------+---------+
|Алексей|    101|        2|
|Алексей|    101|        1|
|  Мария|    102|        2|
|   Иван|    103|        1|
|  Елена|    104|        1|
|  Елена|    104|        3|
|Дмитрий|    105|       99|
+-------+-------+---------+



In [20]:
(
    df_users.alias("u")
        .join(df_enrollments.alias("e"), "user_id", "left")
        .filter("e.course_id IS NULL")
        .show()
)

+-------+----+----+---------+----+
|user_id|city|name|course_id|date|
+-------+----+----+---------+----+
+-------+----+----+---------+----+



In [25]:
from pyspark.sql.functions import sum, when

(
    df_users.alias("u")
        .join(df_enrollments.alias("e"), df_users["user_id"] == df_enrollments["user_id"], "left")
        .groupBy(df_users["user_id"])
        .agg(sum(when(df_enrollments["course_id"].isNull(), 0).otherwise(1)).alias("course_count"))
        .orderBy("u.user_id")
        .show()
)

+-------+------------+
|user_id|course_count|
+-------+------------+
|    101|           2|
|    102|           1|
|    103|           1|
|    104|           2|
|    105|           1|
+-------+------------+



In [26]:
spark.stop()